# SQL

In [1]:
SQLITE_PATH = './data/dbfinal_2.sqlite3'
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def execute_query(db, query):
    conn = sqlite3.connect(db)
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    conn.commit()
    conn.close()
    return rows

In [ ]:
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE white_elo < 1700 OR white_elo >= 1800;")
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE type = 'Bullet' OR type = 'UltraBullet' OR type = 'Correspondence';")
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE low_time = 1;")

In [ ]:
execute_query(SQLITE_PATH, "VACUUM;")

[]

In [ ]:
drop_columns = ['result', 'white_player', 'black_player', 'black_elo', 'time_control', 'termination', 'white_won', 'black_won', 'no_winner', 'num_ply', 'winrate', 'winrate_elo', 'winrate_loss', 'is_blunder_wr', 'opp_winrate', 'active_elo', 'opponent_elo', 'active_won', 'is_capture', 'clock', 'opp_clock', 'clock_percent', 'opp_clock_percent', 'active_bishop_count', 'active_knight_count','active_pawn_count','active_queen_count','active_rook_count', 'opp_bishop_count', 'opp_knight_count','opp_pawn_count','opp_queen_count','opp_rook_count', 'type', 'white_elo', 'move_ply', 'cp', 'cp_rel', 'low_time', 'num_legal_moves']

In [ ]:
for column in drop_columns:
    execute_query(SQLITE_PATH, "ALTER TABLE moves DROP COLUMN {};".format(column))
execute_query(SQLITE_PATH, "VACUUM;")

[]

In [6]:
execute_query(SQLITE_PATH, "PRAGMA table_info(moves);")

[(0, 'move', 'TEXT', 0, None, 0),
 (1, 'cp_loss', 'REAL', 0, None, 0),
 (2, 'is_blunder_cp', 'INTEGER', 0, None, 0),
 (3, 'white_active', 'INTEGER', 0, None, 0),
 (4, 'board', 'TEXT', 0, None, 0),
 (5, 'is_check', 'INTEGER', 0, None, 0)]

In [7]:
execute_query(SQLITE_PATH, "SELECT COUNT(*) FROM moves;")

[(9581626,)]

# Create Board Data - One Hot

In [2]:
data = pd.DataFrame(execute_query(SQLITE_PATH, "SELECT * FROM moves;"))
data.rename(columns={0:'move', 1:'cp_loss', 2:'is_blunder_cp', 3:'white_active', 4:'board_pgn', 5:'is_check'}, inplace=True)

In [3]:
data

,move,cp_loss,is_blunder_cp,white_active,board_pgn,is_check
0,e2e4,-0.02,0,1,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0
1,e7e6,0.00,0,0,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,0
2,g1f3,0.03,0,1,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0
3,a7a6,0.39,0,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/5N2/PPPP1PPP/RNBQK...,0
4,d2d4,0.21,0,1,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0
...,...,...,...,...,...,...
9581621,g1g2,1.89,0,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,1
9581622,f3h4,3.35,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5PK1/R1R5...,0
9581623,g2g3,2.06,1,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,1
9581624,h4g6,inf,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP1KP/5P2/R1R5 ...,0


## Add castle rights and ply

In [4]:
def get_castle_from_board(boards):
    list_splitted = [xi.split(sep=' ') for xi in boards]
    return [row[2] for row in list_splitted]

def get_rights_column(right, data):
    rights = []
    for r in data:
        if right in r:
            rights.append(1)
        else:
            rights.append(0)
    return rights

def add_castle_rights(data):
    rights = get_castle_from_board(data['board_pgn'])
    data['white_king_rights'] = get_rights_column('K', rights)
    data['white_queen_rights'] = get_rights_column('Q', rights)
    data['black_king_rights'] = get_rights_column('k', rights)
    data['black_queen_rights'] = get_rights_column('q', rights)

def separate_ply(data):
    data['ply'] = [board.split(sep=' ')[-1] for board in data['board_pgn']]

In [5]:
add_castle_rights(data)
separate_ply(data)
data

,move,cp_loss,is_blunder_cp,white_active,board_pgn,is_check,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights,ply
0,e2e4,-0.02,0,1,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,1,1,1,1,1
1,e7e6,0.00,0,0,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,0,1,1,1,1,1
2,g1f3,0.03,0,1,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,1,1,1,1,2
3,a7a6,0.39,0,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/5N2/PPPP1PPP/RNBQK...,0,1,1,1,1,2
4,d2d4,0.21,0,1,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,1,1,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...
9581621,g1g2,1.89,0,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,1,0,0,0,0,27
9581622,f3h4,3.35,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5PK1/R1R5...,0,0,0,0,0,27
9581623,g2g3,2.06,1,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,1,0,0,0,0,28
9581624,h4g6,inf,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP1KP/5P2/R1R5 ...,0,0,0,0,0,28


## Board to One-hot

In [6]:
piece_to_one_hot_white = {
    'r':[-1.,0.,0.,0.,0.,0.,1.],
    'n':[0.,-1.,0.,0.,0.,0.,1.],
    'b':[0.,0.,-1.,0.,0.,0.,1.],
    'q':[0.,0.,0.,-1.,0.,0.,1.],
    'k':[0.,0.,0.,0.,-1.,0.,1.],
    'p':[0.,0.,0.,0.,0.,-1.,1.],
    'R':[1.,0.,0.,0.,0.,0.,1.],
    'N':[0.,1.,0.,0.,0.,0.,1.],
    'B':[0.,0.,1.,0.,0.,0.,1.],
    'Q':[0.,0.,0.,1.,0.,0.,1.],
    'K':[0.,0.,0.,0.,1.,0.,1.],
    'P':[0.,0.,0.,0.,0.,1.,1.]
}
empty_square_white = [0.,0.,0.,0.,0.,0.,1.]

piece_to_one_hot_black = {
    'r':[-1.,0.,0.,0.,0.,0.,0],
    'n':[0.,-1.,0.,0.,0.,0.,0],
    'b':[0.,0.,-1.,0.,0.,0.,0],
    'q':[0.,0.,0.,-1.,0.,0.,0],
    'k':[0.,0.,0.,0.,-1.,0.,0],
    'p':[0.,0.,0.,0.,0.,-1.,0],
    'R':[1.,0.,0.,0.,0.,0.,0],
    'N':[0.,1.,0.,0.,0.,0.,0],
    'B':[0.,0.,1.,0.,0.,0.,0],
    'Q':[0.,0.,0.,1.,0.,0.,0],
    'K':[0.,0.,0.,0.,1.,0.,0],
    'P':[0.,0.,0.,0.,0.,1.,0]
}
empty_square_black = [0.,0.,0.,0.,0.,0.,0]

In [7]:
def get_one_hot(board: str):
    fen_board = board.split(sep=' ')[0]
    active = board.split(sep=' ')[1]
    one_hot_board = []
    rows = fen_board.split(sep='/')
    for row in rows:
        one_hot_row = []
        for piece in [*row]:
            if piece.isdigit():
                for i in range(int(piece)):
                    if active == 'w':
                        one_hot_row.append(empty_square_white)
                    else:
                        one_hot_row.append(empty_square_black)
            else:
                if active == 'w':
                    one_hot_row.append(piece_to_one_hot_white[piece])
                else:
                    one_hot_row.append(piece_to_one_hot_black[piece])
        one_hot_board.append(one_hot_row)
    return one_hot_board

def boards_to_one_hot(data):
    data['board'] = data['board_pgn'].apply(get_one_hot)

In [8]:
boards_to_one_hot(data)
data

,move,cp_loss,is_blunder_cp,white_active,board_pgn,is_check,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights,ply,board
0,e2e4,-0.02,0,1,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,1,1,1,1,1,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ..."
1,e7e6,0.00,0,0,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,0,1,1,1,1,1,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, -1..."
2,g1f3,0.03,0,1,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,1,1,1,1,2,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ..."
3,a7a6,0.39,0,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/5N2/PPPP1PPP/RNBQK...,0,1,1,1,1,2,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, -1..."
4,d2d4,0.21,0,1,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,1,1,1,1,3,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
9581621,g1g2,1.89,0,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,1,0,0,0,0,27,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0..."
9581622,f3h4,3.35,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5PK1/R1R5...,0,0,0,0,0,27,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, 0.0..."
9581623,g2g3,2.06,1,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,1,0,0,0,0,28,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0..."
9581624,h4g6,inf,1,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP1KP/5P2/R1R5 ...,0,0,0,0,0,28,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, 0.0..."


## Separar movimiento en posicion inicial y final

In [9]:
data['initial_pos'] = [move[:2] for move in data['move']]
data['final_pos'] = [move[2:4] for move in data['move']]

In [10]:
data['promotion'] = [move[4] if len(move) == 5 else 'none' for move in data['move']]
data['move'] = [move[:4] for move in data['move']]

In [11]:
data = data[['board', 'board_pgn', 'ply', 'move', 'initial_pos', 'final_pos', 'promotion', 'is_blunder_cp', 'is_check', 'cp_loss', 'white_active']]
data

,board,board_pgn,ply,move,initial_pos,final_pos,promotion,is_blunder_cp,is_check,cp_loss,white_active
0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ...",rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,e2e4,e2,e4,none,0,0,-0.02,1
1,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, -1...",rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,1,e7e6,e7,e6,none,0,0,0.00,0
2,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ...",rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,2,g1f3,g1,f3,none,0,0,0.03,1
3,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, -1...",rnbqkbnr/pppp1ppp/4p3/8/4P3/5N2/PPPP1PPP/RNBQK...,2,a7a6,a7,a6,none,0,0,0.39,0
4,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, ...",rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,3,d2d4,d2,d4,none,0,0,0.21,1
...,...,...,...,...,...,...,...,...,...,...,...
9581621,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0...",2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,27,g1g2,g1,g2,none,0,1,1.89,1
9581622,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, 0.0...",2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5PK1/R1R5...,27,f3h4,f3,h4,none,1,0,3.35,0
9581623,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0...",2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,28,g2g3,g2,g3,none,1,1,2.06,1
9581624,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0], [0.0, 0.0...",2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP1KP/5P2/R1R5 ...,28,h4g6,h4,g6,none,1,0,inf,0


In [12]:
train_val_df, test_df = train_test_split(data, test_size=0.1, random_state=47, stratify=data['white_active'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=47, stratify=train_val_df['white_active'])

In [13]:
print("Train: {}".format(len(train_df)))
print("Validation: {}".format(len(val_df)))
print("Test: {}".format(len(test_df)))

Train: 7761116
Validation: 862347
Test: 958163


## Almacenar en un csv

In [14]:
train_df.to_csv("./data/final_data_3/ml_final_data_train.csv")
val_df.to_csv("./data/final_data_3/ml_final_data_val.csv")
test_df.to_csv("./data/final_data_3/ml_final_data_test.csv")

### Almacenar en BD

In [24]:
# SQLITE_ML_PATH = './data/db_ml_final.sqlite3'

# def serialize_board(board):
#     return np.array(board).tobytes()

# def create_final_db_ml(data):
#     conn = sqlite3.connect(SQLITE_ML_PATH)
#     cursor = conn.cursor()
#     cursor.execute("""
#     CREATE TABLE IF NOT EXISTS moves (
#         board BLOB,
#         white_king_rights INTEGER,
#         white_queen_rights INTEGER,
#         black_king_rights INTEGER,
#         black_queen_rights INTEGER,
#         initial_pos TEXT,
#         final_pos TEXT,
#         is_blunder_cp INTEGER,
#         is_check INTEGER
#     )
#     """)
#     conn.commit()

#     for index, row in data.iterrows():
#         board_blob = serialize_board(row['board'])
#         cursor.execute("INSERT INTO moves (board, white_king_rights, white_queen_rights, black_king_rights, black_queen_rights, initial_pos, final_pos, is_blunder_cp, is_check) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", (board_blob, row['white_king_rights'], row['white_queen_rights'], row['black_king_rights'], row['black_queen_rights'], row['initial_pos'], row['final_pos'], row['is_blunder_cp'], row['is_check']))
    
#     conn.commit()
#     conn.close()


In [25]:
# create_final_db_ml(data)